# ETL Raw - Silver

**Dataset:** Uber Ride Analytics - Nova Delhi, Índia  
**Fonte:** https://www.kaggle.com/datasets/yashdevladdha/uber-ride-analytics-dashboard

### Importações e Configurações do Postgres


In [1]:
import pandas as pd
import numpy as np
import psycopg2
import os
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# Configurações do PostgreSQL
DB_NAME = "uber_analytics_silver"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"

### Extração dos dados:

In [2]:
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
csv_path = os.path.join(
    project_root,
    "Data Layer",
    "raw",
    "dados_brutos.csv"
)
df = pd.read_csv(csv_path)

print(f"Total de registros: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")
print(f"Colunas: {df.columns}")

Total de registros: 150000
Total de colunas: 21
Colunas: Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT',
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Payment Method'],
      dtype='object')


## Transformar

### Renomeando e preparando colunas

In [3]:
# Copiar dados originais
df_tratado = df.copy()
df_tratado = df_tratado.rename(columns={
    'Booking ID': 'id',
    'Booking Status': 'status',
    'Vehicle Type': 'vehicle',
    'Pickup Location': 'pickup',
    'Drop Location': 'drop',
    'Avg VTAT': 'vtat',
    'Avg CTAT': 'ctat',
    'Customer ID': 'customer_id',
    'Cancelled Rides by Customer': 'cancelled_by_customer',
    'Cancelled Rides by Driver': 'cancelled_by_driver',
    'Reason for cancelling by Customer': 'reason_cancelled_by_customer',
    'Driver Cancellation Reason': 'reason_cancelled_by_driver',
    'Incomplete Rides': 'incomplete',
    'Incomplete Rides Reason': 'reason_incomplete',
    'Booking Value': 'value',
    'Ride Distance': 'distance',
    'Payment Method': 'payment',
    'Customer Rating': 'customer_rating',
    'Driver Ratings': 'driver_rating',
    'Date': 'date',
    'Time': 'time',
})

if 'date' in df_tratado.columns:
    df_tratado['date'] = pd.to_datetime(df_tratado['date'], errors='coerce')
if 'time' in df_tratado.columns:
    df_tratado['time'] = pd.to_datetime(df_tratado['time'], format='%H:%M:%S', errors='coerce').dt.time

numeric_columns = ['vtat', 'ctat', 'value', 'distance', 'driver_rating', 'customer_rating']
for col in numeric_columns:
    if col in df_tratado.columns:
        df_tratado[col] = pd.to_numeric(df_tratado[col], errors='coerce')

boolean_columns = ['cancelled_by_customer', 'cancelled_by_driver', 'incomplete']
for col in boolean_columns:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].map({
            'TRUE': True, 'True': True, 'true': True, 1: True, '1': True,
            'FALSE': False, 'False': False, 'false': False, 0: False, '0': False
        })

### Removendo duplicatas e tratando nulos

In [4]:

df_tratado = df_tratado.drop_duplicates(subset=['id'], keep='first')
df_tratado = df_tratado.dropna(subset=['id'])
df_tratado = df_tratado.dropna(subset=['customer_id'])

for col in boolean_columns:
    if col in df_tratado.columns:
        df_tratado[col] = (
            df_tratado[col]
            .astype("boolean")
            .fillna(False)
            .astype(bool)
        )

text_columns = [
    'reason_cancelled_by_customer',
    'reason_cancelled_by_driver',
    'reason_incomplete'
]
for col in text_columns:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].fillna('')


### Validação de dados

In [5]:
if 'driver_rating' in df_tratado.columns:
    df_tratado.loc[(df_tratado['driver_rating'] < 1) | (df_tratado['driver_rating'] > 5), 'driver_rating'] = np.nan

if 'customer_rating' in df_tratado.columns:
    df_tratado.loc[(df_tratado['customer_rating'] < 1) | (df_tratado['customer_rating'] > 5), 'customer_rating'] = np.nan

if 'value' in df_tratado.columns:
    df_tratado.loc[df_tratado['value'] < 0, 'value'] = np.nan

if 'distance' in df_tratado.columns:
    df_tratado.loc[df_tratado['distance'] < 0, 'distance'] = 0

### Padronização de categorias

In [6]:
if 'status' in df_tratado.columns:
    df_tratado['status'] = df_tratado['status'].str.strip().str.title()


if 'vehicle' in df_tratado.columns:
    df_tratado['vehicle'] = df_tratado['vehicle'].str.strip().str.title()


if 'payment' in df_tratado.columns:
    df_tratado['payment'] = df_tratado['payment'].str.strip().str.title()


for col in ['pickup', 'drop']:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].str.strip().str.title()

### Selecionando apenas as colunas desejadas( caso haja atualização da base de dados)

In [7]:
colunas_finais = [
    'id', 'date', 'time', 'status', 'customer_id', 'vehicle',
    'pickup', 'drop',
    'vtat', 'ctat',
    'cancelled_by_customer', 'reason_cancelled_by_customer',
    'cancelled_by_driver', 'reason_cancelled_by_driver',
    'incomplete', 'reason_incomplete',
    'value', 'distance',
    'driver_rating', 'customer_rating',
    'payment'
]

colunas_existentes = [col for col in colunas_finais if col in df_tratado.columns]

df_final = df_tratado[colunas_existentes].copy()
df_final = df_final.reindex(columns=colunas_existentes)

print("Formato do DataFrame final:")
print(f"Total de registros: {len(df_final)}")
print(f"Total de colunas: {len(df_final.columns)}")
print("\nColunas:")
for i, col in enumerate(df_final.columns, 1):
    print(f"{i:2}. {col}")

Formato do DataFrame final:
Total de registros: 148767
Total de colunas: 21

Colunas:
 1. id
 2. date
 3. time
 4. status
 5. customer_id
 6. vehicle
 7. pickup
 8. drop
 9. vtat
10. ctat
11. cancelled_by_customer
12. reason_cancelled_by_customer
13. cancelled_by_driver
14. reason_cancelled_by_driver
15. incomplete
16. reason_incomplete
17. value
18. distance
19. driver_rating
20. customer_rating
21. payment


## Carregar no PostgreSQL

### Criando e Conectando com o banco PostgreSQL

In [8]:
def criar_banco_se_nao_existir():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT,
            database='postgres'
        )
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cursor = conn.cursor()

        
        cursor.execute(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'")
        existe = cursor.fetchone()

        if not existe:
            cursor.execute(f"CREATE DATABASE {DB_NAME}")
            print(f"Banco de dados '{DB_NAME}' criado com sucesso!")
        else:
            print(f"Banco de dados '{DB_NAME}' já existe.")

        cursor.close()
        conn.close()
        return True

    except Exception as e:
        print(f"Erro ao criar banco: {e}")
        return False


if criar_banco_se_nao_existir():
    print(f"\nConectando ao banco '{DB_NAME}'...")

    conexao = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT
    )

    cursor = conexao.cursor()
    print("Conexão efetuada com sucesso!")

Erro ao criar banco: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?



### Criando tabela BOOKING no PostgreSQL

In [9]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS booking (
    id VARCHAR(50) PRIMARY KEY,
    date DATE,
    time TIME,
    status VARCHAR(50),
    customer_id VARCHAR(50),
    vehicle VARCHAR(50),
    pickup VARCHAR(100),
    drop VARCHAR(100),
    vtat DECIMAL(10,2),
    ctat DECIMAL(10,2),
    cancelled_by_customer BOOLEAN,
    reason_cancelled_by_customer TEXT,
    cancelled_by_driver BOOLEAN,
    reason_cancelled_by_driver TEXT,
    incomplete BOOLEAN,
    reason_incomplete TEXT,
    value DECIMAL(10,2),
    distance DECIMAL(10,2),
    driver_rating DECIMAL(3,2),
    customer_rating DECIMAL(3,2),
    payment VARCHAR(50)
)
"""

try:
    cursor.execute(create_table_sql)
    conexao.commit()
except Exception as e:
    print(f"Erro ao criar tabela: {e}")
    conexao.rollback()

Erro ao criar tabela: name 'cursor' is not defined


NameError: name 'conexao' is not defined

### Limpando tabela caso exista

In [10]:
try:
    cursor.execute("TRUNCATE TABLE booking RESTART IDENTITY")
    conexao.commit()
except Exception as e:
    print(f"Erro ao limpar tabela: {e}")
    conexao.rollback()

Erro ao limpar tabela: name 'cursor' is not defined


NameError: name 'conexao' is not defined

### Preparando dados para inserção

In [11]:
def preparar_valor(valor):
    if pd.isna(valor):
        return None
    elif isinstance(valor, bool):
        return bool(valor)
    elif isinstance(valor, (int, np.integer)):
        return int(valor)
    elif isinstance(valor, (float, np.floating)):
        return float(valor)
    elif isinstance(valor, pd.Timestamp):
        return valor.strftime('%Y-%m-%d')
    elif isinstance(valor, pd._libs.tslibs.time.Time):
        return str(valor)
    else:
        return str(valor)


dados_para_inserir = []

for _, row in df_final.iterrows():
    valores = []
    for col in df_final.columns:
        valores.append(preparar_valor(row[col]) if col in row else None)
    dados_para_inserir.append(tuple(valores))

print(f"\nDados preparados: {len(dados_para_inserir):,} registros prontos para inserção")

AttributeError: module 'pandas._libs.tslibs' has no attribute 'time'

### Inserindo dados em lote

In [12]:
placeholders = ', '.join(['%s'] * len(df_final.columns))
colunas_sql = ', '.join(df_final.columns)
insert_sql = f"INSERT INTO booking ({colunas_sql}) VALUES ({placeholders})"



batch_size = 1000
total_inseridos = 0
for i in range(0, len(dados_para_inserir), batch_size):
    batch = dados_para_inserir[i:i + batch_size]
    cursor.executemany(insert_sql, batch)
    conexao.commit()
    total_inseridos += len(batch)

    if (i + batch_size) % 5000 == 0 or i + batch_size >= len(dados_para_inserir):
        print(f"  Inseridos: {total_inseridos:,} / {len(dados_para_inserir):,} registros...")

### Fechando a conexão.

In [13]:

cursor.close()
conexao.close()


NameError: name 'cursor' is not defined